In [1]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

# SAM 3 Agent

This notebook shows an example of how an MLLM can use SAM 3 as a tool, i.e., "SAM 3 Agent", to segment more complex text queries such as "the leftmost child wearing blue vest".

## Env Setup

First install `sam3` in your environment using the [installation instructions](https://github.com/facebookresearch/sam3?tab=readme-ov-file#installation) in the repository.

In [2]:
import torch
# turn on tfloat32 for Ampere GPUs
# https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# use bfloat16 for the entire notebook. If your card doesn't support it, try float16 instead
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

# inference mode for the whole notebook. Disable if you need gradients
torch.inference_mode().__enter__()

In [3]:
import os
import shutil

SAM3_ROOT = os.path.expanduser("~/run/JiaBSH/sam3")
if not os.path.isdir(SAM3_ROOT):
    raise FileNotFoundError(f"SAM3_ROOT does not exist: {SAM3_ROOT}")

print(f"Changing working directory to {SAM3_ROOT}")
os.chdir(SAM3_ROOT)

# setup GPU to use - a single GPU is enough for this demo
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if shutil.which("nvidia-smi") is not None:
    _ = os.system("nvidia-smi")
else:
    print("nvidia-smi not found; skipping GPU status check")

Changing working directory to /data/home/scvi576/run/JiaBSH/sam3
Tue May 19 16:09:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:D8:00.0 Off |                  N/A |
|  0%   31C    P8             15W /  600W |       3MiB /  32607MiB |      0%      Default |
|                                         |                        |       

## Build SAM3 Model

In [4]:
import sam3
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

sam3_root = os.path.dirname(sam3.__file__)
ckpt = "/data/home/scvi576/run/JiaBSH/sam3/ms_cache/facebook/sam3/sam3.pt" # <-- change to your local file

model = build_sam3_image_model(
    checkpoint_path=ckpt,
    load_from_HF=False,
    device="cuda" if torch.cuda.is_available() else "cpu",
    eval_mode=True,
)
processor = Sam3Processor(model, confidence_threshold=0.5)

/data/run01/scvi576/envs/sam3/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## LLM Setup

Config which MLLM to use, it can either be a model served by vLLM that you launch from your own machine or a model is served via external API. If you want to using a vLLM model, we also provided insturctions below.

In [5]:
LLM_CONFIGS = {
   
    "qwen3.5-27b": {
        "provider": "dashscope",
        "base_url": "https://dashscope.aliyuncs.com/compatible-mode/v1"
    },
    # models served via external APIs
    # add your own
}

model = "qwen3.5-27b"
LLM_API_KEY = 'sk-ee6ba686614c4bd89768324ce4163f5a'

llm_config = LLM_CONFIGS[model]
llm_config["api_key"] = LLM_API_KEY
llm_config["name"] = model
llm_config["model"] = model  # keep both keys for compatibility with downstream calls

# setup API endpoint
if llm_config["provider"] == "vllm":
    LLM_SERVER_URL = "http://0.0.0.0:8001/v1"  # replace this with your vLLM server address as needed
else:
    LLM_SERVER_URL = llm_config["base_url"]

### Setup vLLM server 
This step is only required if you are using a model served by vLLM, skip this step if you are calling LLM using an API like Gemini and GPT.

* Install vLLM (in a separate conda env from SAM 3 to avoid dependency conflicts).
  ```bash
    conda create -n vllm python=3.12
    pip install vllm --extra-index-url https://download.pytorch.org/whl/cu128
  ```
* Start vLLM server on the same machine of this notebook
  ```bash
    # qwen 3 VL 8B thinking
    vllm serve Qwen/Qwen3-VL-8B-Thinking --tensor-parallel-size 4 --allowed-local-media-path / --enforce-eager --port 8001
  ```

## Run SAM3 Agent Inference

In [6]:
from functools import partial
from IPython.display import display, Image
from sam3.agent.client_llm import send_generate_request as send_generate_request_orig
from sam3.agent.client_sam3 import call_sam_service as call_sam_service_orig
from sam3.agent.inference import run_single_image_inference

In [8]:
# prepare input args and run single image inference
image = "mmdata_test_1024/2_5x_unsup/image/2.5x-1.png"
#prompt = "the leftmost child wearing blue vest"
prompt = "识别出图像中的所有石墨烯畴区，每个石墨烯畴区都是六边形，特别注意多个石墨烯畴区可能会重叠在一起，你需要将他们区分成多个实例"
image = os.path.abspath(image)
selected_model = llm_config.get("model", llm_config.get("name"))
if selected_model is None:
    raise KeyError("llm_config must contain either 'model' or 'name'")
send_generate_request = partial(
    send_generate_request_orig,
    server_url=LLM_SERVER_URL,
    model=selected_model,
    api_key=llm_config["api_key"],
)
call_sam_service = partial(call_sam_service_orig, sam3_processor=processor)
output_image_path = run_single_image_inference(
    image, prompt, llm_config, send_generate_request, call_sam_service,
    debug=True, output_dir="agent_output"
 )

# display output
if output_image_path is not None:
    display(Image(filename=output_image_path))

------------------------------ Starting SAM 3 Agent Session... ------------------------------ 
> Text prompt: 识别出图像中的所有石墨烯畴区，每个石墨烯畴区都是六边形，特别注意多个石墨烯畴区可能会重叠在一起，你需要将他们区分成多个实例
> Image path: /data/run01/scvi576/JiaBSH/sam3/mmdata_test_1024/2_5x_unsup/image/2.5x-1.png



------------------------------ Round 1------------------------------



image_path /data/run01/scvi576/JiaBSH/sam3/mmdata_test_1024/2_5x_unsup/image/2.5x-1.png
🔍 Calling model qwen3.5-27b...


KeyboardInterrupt: 